# Volcano plots
Volcano Plot in Undifferentiated Pleomorphic Sarcoma (UPS, n = 26)

This notebook visualizes differential gene expression between the C1 (n = 21) and C2 (n = 5) subgroups in a dataset of 26 UPS samples, using generated DESeq2 results.

Purpose:
1,distribution of differentially expressed genes between C1 and C2.
2,Highlight significant genes belonging to selected GSEA gene sets (GO Molecular Function).
3,Label the top 5 most significant genes.

## Requirements
- input data

1，DESeq2 Results
The input file contains previously generated DESeq2 differential expression results comparing the C1 and C2 subgroups of 26 
UPS samples.

Input file:../results/deseq2/DESeq2_all_26samples.tsv

Required columns: 
gene_name: Gene symbols used for gene identification and labeling.
log2FoldChange: Log2 fold change of gene expression between C2 and C1.
padj: Adjusted p-values used to determine statistical significance.

Genes with padj < 0.05 are considered significantly differentially expressed.

2，GSEA Gene Set Results
The input files contain gene lists from selected GO Molecular Function (GOMF) gene sets obtained from previous GSEA analysis.

Input directory:../results/GSEA_26/GOMF_geneset.Gsea.1791350189868/

Selected gene sets:
C1:
GOMF_IMMUNE_RECEPTOR_ACTIVITY
GOMF_CYTOKINE_RECEPTOR_ACTIVITY
GOMF_CHEMOKINE_ACTIVITY
C2:
GOMF_DNA_HELICASE_ACTIVITY
GOMF_DNA_BINDING_BENDING
GOMF_DAMAGED_DNA_BINDING

Required column: SYMBOL: Gene symbols used to match genes in the DESeq2 results.

- required software
This notebook was run using R 4.5.3 in a Jupyter Notebook environment (IRkernel 1.3.2) on macOS Tahoe 26.6.2.
The following R packages were used:
ggplot(4.0.3):Used to create volcano plots.
ggrepel(0.9.8):Used to label genes while minimizing text overlap.
svglite(2.2.2):Used to export plots in SVG format.

## Results
1，What important information do I gain from this analysis?
Identify significantly differentially expressed genes between C1 and C2.
Identify significant genes within selected GOMF gene sets and label the top 5 most significant genes.

2，What plots, files etc. are created and where?

Plots:
Overview volcano plots (grey and colored).
Six GOMF gene set volcano plots.

Output directory: ../results/

File formats: PNG and SVG.

In [ ]:
# Input Data — CHANGE HERE when using new samples
df<-read.delim("../results/deseq2/DESeq2_all_26samples.tsv")

# Preview the data
head(df)

In [ ]:
# Check data dimensions and column names
nrow(df)
ncol(df)
colnames(df)

In [ ]:
library(ggplot2)
library(ggrepel)

# Remove genes with missing or invalid adjusted p-values
df_plot <- df[!is.na(df$padj) & df$padj > 0, ]

# Axis Settings
# Change here to adjust the axes of all volcano plots
volcano_axes <- list(scale_x_continuous(breaks = seq(-10, 10, 5)),scale_y_continuous(breaks = seq(0, 10, 5)),coord_cartesian(xlim = c(-10, 10), ylim = c(0, 10)))

# Create grey volcano plot
ggplot(df_plot, aes(x = log2FoldChange, y = -log10(padj))) +
  geom_point(color = "grey50", size = 1, alpha = 0.6) +
  theme_classic() +
  labs(title = "Volcano Plot: C1 vs C2 (26 samples)",x = "log2 Fold Change",y = "-log10(adjusted p-value)",caption="Note:Y-axis truncated at 10.")+
  volcano_axes

In [ ]:
# Classify genes
df_plot$group <- "Not significant"
df_plot$group[df_plot$padj < 0.05 & df_plot$log2FoldChange < 0] <- "C1 higher"
df_plot$group[df_plot$padj < 0.05 & df_plot$log2FoldChange > 0] <- "C2 higher"

# Volcano plot with colors
ggplot(df_plot, 
  aes(x = log2FoldChange, y = -log10(padj))) +
  geom_point(aes(color = group), size = 1, alpha = 0.6) +
  scale_color_manual(values = c("C1 higher" = "#4285B4","C2 higher" = "#DC7050","Not significant" = "grey70")) +
  theme_classic() +
  labs(
  title = "Volcano Plot: C1 vs C2 (26 samples)",
  x = "log2 Fold Change (C2 / C1)",
  y = "-log10(adjusted p-value)",
  color = "Gene group")+
  volcano_axes

In [ ]:
# Define a reusable function to highlight selected gene sets on a volcano plot
volcano_plot_gene_set <- function(df_plot, gene_set, plot_title = "Volcano Plot") {
 
   # Significant genes in the selected gene set
     highlight_df <- subset(df_plot, gene_name %in% gene_set & group != "Not significant")

   # Top 5 most significant genes for labeling
     label_df <- highlight_df[order(highlight_df$padj), ]
     label_df <- head(label_df, 5)
    
   #Create volcano plot
     p<- ggplot(df_plot, aes(x = log2FoldChange, y = -log10(padj))) +
         geom_point(aes(color = group), size = 1, alpha = 0.5) +
         scale_color_manual(values = c("C1 higher" = "#4285B4","C2 higher" = "#DC7050","Not significant" = "grey70")) +
     theme_classic() +
     labs(title = plot_title,x = "log2 Fold Change (C2 / C1)",y = "-log10(adjusted p-value)",color = "Gene group")+
     volcano_axes
   # Highlighted gene-set members and labels
     p+
     geom_point(data = highlight_df,shape = 21,fill = NA,color = "black",size = 3,stroke = 1) +
     geom_text_repel(data = label_df,aes(label = gene_name),size = 3,color = "black",box.padding = 0.3,point.padding = 0.2,max.overlaps = Inf)
}

In [ ]:
# Define a function to plot GSEA gene sets automatically
# Note: Change gsea_dir when analyzing GO BP or Hallmark gene sets.

gsea_dir <- "../results/GSEA_26/GOMF_geneset.Gsea.1791350189868"

volcano_plot_gsea <- function(gene_set, cluster) {
  #Read genes from the GSEA result
   gene_file <- file.path(gsea_dir, paste0(gene_set, ".tsv"))
   genes <- read.delim(gene_file)$SYMBOL
  #Create plot title  
   name <- sub("^GOMF_", "", gene_set)
   plot_title<-paste0(cluster, ": ", gsub("_", " ", name))
  #Generate volcano plot
  p<-volcano_plot_gene_set(df_plot,genes,plot_title)

p

}

In [ ]:
# C1: Immune Receptor Activity
volcano_plot_gsea("GOMF_IMMUNE_RECEPTOR_ACTIVITY", "C1")

In [ ]:
volcano_plot_gsea("GOMF_CYTOKINE_RECEPTOR_ACTIVITY", "C1")

In [ ]:
volcano_plot_gsea("GOMF_CHEMOKINE_ACTIVITY", "C1")

In [ ]:
volcano_plot_gsea("GOMF_DNA_HELICASE_ACTIVITY", "C2")

In [ ]:
volcano_plot_gsea("GOMF_DNA_BINDING_BENDING", "C2")

In [ ]:
volcano_plot_gsea("GOMF_DAMAGED_DNA_BINDING", "C2")

In [ ]:
# GSEA Volcano Plots — Final Figures

plots <- list(
  C1_IMMUNE_RECEPTOR_ACTIVITY = volcano_plot_gsea("GOMF_IMMUNE_RECEPTOR_ACTIVITY", "C1"),
  C1_CYTOKINE_RECEPTOR_ACTIVITY = volcano_plot_gsea("GOMF_CYTOKINE_RECEPTOR_ACTIVITY", "C1"),
  C1_CHEMOKINE_ACTIVITY = volcano_plot_gsea("GOMF_CHEMOKINE_ACTIVITY", "C1"),
  C2_DNA_HELICASE_ACTIVITY = volcano_plot_gsea("GOMF_DNA_HELICASE_ACTIVITY", "C2"),
  C2_DNA_BINDING_BENDING = volcano_plot_gsea("GOMF_DNA_BINDING_BENDING", "C2"),
  C2_DAMAGED_DNA_BINDING = volcano_plot_gsea("GOMF_DAMAGED_DNA_BINDING", "C2")
)

In [ ]:
# Save GSEA Volcano Plots — PNG

for (name in names(plots)) {
  ggsave(
    filename = file.path("../results", paste0(name, ".png")),
    plot = plots[[name]],
    width = 7,
    height = 5,
    dpi = 300
  )
}

In [ ]:
# Save GSEA Volcano Plots — SVG

for (name in names(plots)) {
  ggsave(
    filename = file.path("../results", paste0(name, ".svg")),
    plot = plots[[name]],
    width = 7,
    height = 5
  )
}